In [ ]:
# constructing the figure for HST/Webb
# nircam time

import dLux as dl
import jax.numpy as np
import jax
import matplotlib.pyplot as plt
import dLux.utils as dlu
from dLux.sources import Source
from jax import Array
from dLux.sources import BaseSource
from jax import vmap
import jax.random as jr
from dLux import spectra
from matplotlib.style import use
use('default')
jax.config.update("jax_enable_x64", True)
plt.rcParams['image.origin'] = 'lower'

from jax.scipy.special import gammaln, xlogy

def poisson_loglike_gamma(k, mu):
    # analytic continuation using gammaln(k+1)
    k = np.asarray(k, dtype=float)
    mu = np.asarray(mu, dtype=float)
    valid = (k >= 0) & np.isfinite(k) & (mu > 0) & np.isfinite(mu)
    logp = xlogy(k, mu) - mu - gammaln(k + 1.0)
    return np.where(valid, logp, -np.inf)

# define the likelihood function:
def poiss_loglike(pytree, data) -> float:
    """
    Poissonian log likelihood of the pytree given the data. Assumes the pytree
    has a .model() function.

    Parameters
    ----------
    pytree : Base
        Pytree with a .model() function.
    data : Array
        Data to compare the model to.

    Returns
    -------
    log_likelihood : Array
        Log likelihood of the pytree given the data.
    """
    return poisson_loglike_gamma(pytree.model(), data).sum()

def normal_logpdf(x, mu, sigma):
    z = (x-mu)/sigma
    return -0.5 * (np.log(2*np.pi) + 2*np.log(sigma) + z**2)


# JATIS style: 8 pt fonts, 5.25 in width, 0.5 pt spines and ticks (see jatis_style.py)
import jatis_style as js
js.apply()

fig_width = js.FIG_WIDTH


In [ ]:
eps = 1e-15
class CombinedSpectrum(dl.Spectrum):
    wavelengths: Array
    filt_weights: Array
    basis_weights: Array
    weights: Array
    flux: float

    def __init__(self, wavels, filt_weights, basis_weights):
        self.wavelengths = np.asarray(wavels, dtype=float)
        self.filt_weights = np.asarray(filt_weights, dtype=float)
        self.basis_weights = np.asarray(basis_weights, dtype=float)

        spec_w = self.spec_weights()
        detected_w = self.filt_weights * spec_w

        self.flux = detected_w.sum()
        self.weights = detected_w / self.flux
        
    def spec_weights(self):
        raise NotImplementedError

    def normalise(self):
        return self


class CombinedBasisSpectrum(CombinedSpectrum):
    basis_vects: Array

    def __init__(self, wavels, filt_weights, basis_weights, basis):
        self.basis_vects = np.asarray(basis, dtype=float)
        super().__init__(wavels, filt_weights, basis_weights)

    def spec_weights(self):
        return np.maximum(np.sum(self.basis_vects * self.basis_weights, axis=1), eps)
        # return 10 ** (np.sum(self.basis_vects * self.basis_weights, axis=1))

In [ ]:
# normal model psf
def model_psf(spectrum, instrument):
    old_spec = instrument.get("source.spectrum")

    new_spec = CombinedBasisSpectrum(
        wavels=old_spec.wavelengths,
        filt_weights=old_spec.filt_weights,
        basis_weights=spectrum,
        basis=old_spec.basis_vects,
    )

    telescope = instrument.set("source.spectrum", new_spec)
    telescope = telescope.set("source.flux", telescope.get("source.spectrum").flux)
    
    return telescope.model()

In [ ]:
import numpy as onp
def bin_filter_midpoints(wave, throughput, n_bins):
    """
    Bin a throughput curve into n_bins equal-width wavelength bins.

    Returns
    -------
    centres : array
        Midpoint wavelength of each bin.
    mean_throughput : array
        Average throughput in each bin.
    edges : array
        Bin edges.
    """
    wave = onp.asarray(wave)
    throughput = onp.asarray(throughput)

    order = onp.argsort(wave)
    wave = wave[order]
    throughput = throughput[order]

    edges = onp.linspace(wave.min(), wave.max(), n_bins + 1)
    centres = 0.5 * (edges[:-1] + edges[1:])

    mean_throughput = onp.zeros(n_bins)

    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]

        mask = (wave >= lo) & (wave <= hi)

        # Include bin edges by interpolation so partial bins behave correctly
        w = onp.concatenate([[lo], wave[mask], [hi]])
        t = onp.interp(w, wave, throughput)

        # Remove duplicated points
        w, unique = onp.unique(w, return_index=True)
        t = t[unique]

        mean_throughput[i] = onp.trapezoid(t, w) / (hi - lo)

    return centres, mean_throughput, edges

In [ ]:
# importing the filters
# choose filter
arr = onp.loadtxt('../nircam_filts/f444w.txt')
wave_um = arr[:, 0]
throughput = arr[:, 1]

wave_m = wave_um * 1e-6

# truncating to remove zeros
through_thresh = 1e-3*np.max(throughput)

waves_trim = wave_m[throughput>through_thresh]
throughput_trim = throughput[throughput>through_thresh]

wave_bin, throughput_bin, _ = bin_filter_midpoints(waves_trim, throughput_trim, 101)
plt.plot(waves_trim, throughput_trim)
plt.plot(wave_bin, throughput_bin)
plt.show()

In [ ]:
from __future__ import annotations

import jax
import jax.numpy as jnp
import equinox as eqx

import dLux as dl
from dLux.psfs import PSF
from dLux.wavefronts import Wavefront


class MappedAngularOpticalSystem(dl.AngularOpticalSystem):
    """
    AngularOpticalSystem variant that maps over wavelengths using jax.lax.map
    rather than eqx.filter_vmap over the full wavelength axis.

    This reduces peak memory by evaluating wavelengths in chunks.

    Parameters
    ----------
    wavelength_batch_size : int | None
        Number of wavelengths to process in parallel per chunk.

        - None: pure lax.map, effectively one wavelength at a time.
        - 1: one wavelength at a time.
        - N: process N wavelengths at a time using vmap internally.
        - 0: equivalent to full vmap, i.e. basically the original memory pattern.
    """

    wavelength_batch_size: int | None = eqx.field(static=True)

    def __init__(
        self,
        wf_npixels: int,
        diameter: float,
        layers,
        psf_npixels: int,
        psf_pixel_scale: float,
        oversample: int = 1,
        wavelength_batch_size: int | None = 8,
    ):
        super().__init__(
            wf_npixels=wf_npixels,
            diameter=diameter,
            layers=layers,
            psf_npixels=psf_npixels,
            psf_pixel_scale=psf_pixel_scale,
            oversample=oversample,
        )

        if wavelength_batch_size is not None:
            wavelength_batch_size = int(wavelength_batch_size)
            if wavelength_batch_size < 0:
                raise ValueError("wavelength_batch_size must be None, 0, or a positive int.")

        self.wavelength_batch_size = wavelength_batch_size

    def propagate(
        self,
        wavelengths,
        offset=None,
        weights=None,
        return_wf: bool = False,
        return_psf: bool = False,
    ):
        """
        Polychromatic propagation using jax.lax.map over wavelength.

        Same semantics as dLux.OpticalSystem.propagate, but with chunked
        wavelength evaluation.
        """
        if return_wf and return_psf:
            raise ValueError(
                "Cannot return both Wavefront and PSF objects. "
                "Choose return_wf=True or return_psf=True."
            )

        wavelengths = jnp.atleast_1d(wavelengths)

        if weights is None:
            weights = jnp.ones_like(wavelengths) / wavelengths.size
        else:
            weights = jnp.atleast_1d(weights)

        if weights.shape != wavelengths.shape:
            raise ValueError(
                f"wavelengths and weights must have the same shape, got "
                f"{wavelengths.shape} and {weights.shape}."
            )

        if offset is None:
            offset = jnp.zeros(2)
        else:
            offset = jnp.asarray(offset)

        if offset.shape != (2,):
            raise ValueError(f"offset must have shape (2,), got {offset.shape}.")

        def prop_one(x):
            wavelength, weight = x

            # Important: return_wf=True here so we can weight amplitudes before
            # taking |E|^2, exactly as in dLux's current implementation.
            wf = self.propagate_mono(
                wavelength=wavelength,
                offset=offset,
                return_wf=True,
            )

            return wf.multiply("phasor", jnp.sqrt(weight))

        wf = jax.lax.map(
            prop_one,
            (wavelengths, weights),
            batch_size=self.wavelength_batch_size,
        )

        if return_wf:
            return wf

        psf = wf.psf.sum(axis=0)

        if return_psf:
            return PSF(psf, wf.pixel_scale.mean())

        return psf
    

from __future__ import annotations

import jax
import jax.numpy as np
from jax import lax
from jax import Array

from dLux.optical_systems import AngularOpticalSystem
from dLux.psfs import PSF


class ScannedAngularOpticalSystem(AngularOpticalSystem):
    """
    Memory-safe AngularOpticalSystem.

    This replaces dLux's default wavelength-vectorised propagation with a
    lax.scan over wavelengths, accumulating the final polychromatic PSF directly.

    Main difference from dLux's default propagate:
        default: propagate all wavelengths -> store wavelength stack -> sum PSFs
        this:    propagate one wavelength -> add weighted PSF -> discard

    This is much more memory safe for large wavelength grids.

    Notes
    -----
    This class intentionally does not support return_wf=True, because returning
    a per-wavelength Wavefront object would require storing the full wavelength
    stack, which defeats the point of the class.
    """

    def propagate(
        self,
        wavelengths: Array,
        offset: Array | None = None,
        weights: Array | None = None,
        return_wf: bool = False,
        return_psf: bool = False,
    ) -> Array | PSF:
        if return_wf:
            raise NotImplementedError(
                "ScannedAngularOpticalSystem does not support return_wf=True, "
                "because that would require storing the full wavelength stack."
            )

        if return_wf and return_psf:
            raise ValueError(
                "Cannot return both Wavefront and PSF objects. Choose one."
            )

        wavelengths = np.atleast_1d(wavelengths)

        if weights is None:
            weights = np.ones_like(wavelengths) / wavelengths.size
        else:
            weights = np.atleast_1d(weights)

        if weights.shape != wavelengths.shape:
            raise ValueError(
                f"wavelengths and weights must have the same shape, got "
                f"{wavelengths.shape} and {weights.shape}."
            )

        if offset is None:
            offset = np.zeros(2)
        else:
            offset = np.asarray(offset)

        if offset.shape != (2,):
            raise ValueError(
                f"offset must have shape (2,), got {offset.shape}."
            )

        # Initialise accumulator with a zero PSF of the correct shape.
        #
        # This does perform one extra monochromatic propagation. That is usually
        # acceptable, and it avoids hard-coding the output shape.
        psf0 = np.zeros_like(
            self.propagate_mono(
                wavelength=wavelengths[0],
                offset=offset,
                return_wf=False,
            )
        )

        def step(psf_acc, x):
            wavelength, weight = x

            psf_i = self.propagate_mono(
                wavelength=wavelength,
                offset=offset,
                return_wf=False,
            )

            # dLux default does:
            #   wf_i.multiply("phasor", sqrt(weight)).psf
            #
            # Since psf = |phasor|^2, this is exactly:
            #   weight * psf_i
            psf_acc = psf_acc + weight * psf_i

            return psf_acc, None

        psf, _ = lax.scan(
            step,
            psf0,
            xs=(wavelengths, weights),
        )

        if return_psf:
            # AngularOpticalSystem.to_focus uses:
            #   true_pixel_scale = self.psf_pixel_scale / self.oversample
            #
            # dLux's default return_psf uses wf.pixel_scale.mean(), which should
            # be this same final angular pixel scale in arcsec units for the
            # AngularOpticalSystem.
            pixel_scale = self.psf_pixel_scale / self.oversample
            return PSF(psf, pixel_scale)

        return psf

In [ ]:
aperture_nircam = np.load("../nircam_aperture_2048.npy")
# plt.imshow(aperture_nircam)

layers_nircam = [
    (
        'aperture',
        dl.TransmissiveLayer(aperture_nircam, normalise=True)
    )
]

psf_npix = 400 #sub400p array
psf_pixel_scale = 0.063 #arcsec per pixel, long wavelength channel
oversample = 4 #should be enough
wf_npix = 2048
diameter = 6.603464
# for some reason scanned truly saves memory but batched doesn't
# so do scanned
optics_nircam = ScannedAngularOpticalSystem(
    wf_npix,
    diameter,
    layers_nircam,
    psf_npix,
    psf_pixel_scale,
    oversample,
)

detector = dl.LayeredDetector([
    (
        'downsample',
        dl.detector_layers.Downsample(kernel_size=1)
    )
])

## time for the source
N = throughput_bin.size

spectrum_model = CombinedBasisSpectrum(wave_bin, throughput_bin, np.ones(N), np.eye(N))

source_model = dl.PointSource(
    spectrum=spectrum_model
)

source_mono = dl.PointSource(
    wavelengths=np.array([4.44e-6]),
    flux=1e6
)

instrument = dl.Telescope(optics_nircam, source_model, detector)
instrument_mono = dl.Telescope(optics_nircam, source_mono, detector)

injected_spectrum = np.ones(N)/N * 1e6

img_full = model_psf(injected_spectrum, instrument)
img_mono = instrument_mono.model()

In [ ]:
plt.imshow(img_full**0.25, cmap='magma')
plt.colorbar()
plt.show()

plt.imshow(img_mono**0.25, cmap='magma')
plt.colorbar()
plt.show()

print(np.max(img_full))

In [ ]:
path = "../UVIS/wfc3_uvis2_f606w.txt"  
arr = onp.loadtxt(path)
wave_A = arr[:, 0]      
throughput = arr[:, 1]    

wave_m = wave_A * 1e-10

# truncating to remove zeros

through_thresh = 1e-3*np.max(throughput)

waves_trim = wave_m[throughput>through_thresh]
throughput_trim = throughput[throughput>through_thresh]

# waves_trim = wave_m[np.argmax(throughput>0)-1:]
# throughput_trim = throughput[np.argmax(throughput>0)-1:]

# waves_trim, throughput_trim, _ = trim_to_regular_grid(waves_trim, throughput_trim)

waves_bin, throughput_bin, _ = bin_filter_midpoints(waves_trim, throughput_trim, 101)

In [ ]:
plt.plot(waves_trim, throughput_trim)
plt.plot(waves_bin, throughput_bin)
plt.axhline(0.25*np.max(throughput_trim))
plt.show()

In [ ]:
class HSTMainAperture(dl.CompoundAperture):
    softening : float
    def __init__(self, transformation=dl.CoordTransform(rotation=np.pi/4), softening=0.25):
        self.normalise = True
        self.transformation = transformation
        self.softening = softening
        self.apertures = {
            "mirror" : dl.CircularAperture(
                radius = 1.2,
                softening=self.softening,
                #normalise=True
            ),
            "spider" : dl.Spider(
                width = 0.022*1.2,#0.038*1.2,
                angles = np.asarray([0, 90, 180, 270]),
                softening=self.softening,
            ),
            "secondary" : dl.CircularAperture(
                radius = 0.330*1.2,
                occulting = True,
                softening = self.softening
            ),
            "pad_1" : dl.CircularAperture(
                radius = 0.065*1.2,
                occulting = True,
                transformation=dl.CoordTransform(
                    translation = (0.8921*1.2, 0),
                ),
                softening = self.softening
            ),
            "pad_2" : dl.CircularAperture(
                radius = 0.065*1.2,
                occulting = True,
                transformation=dl.CoordTransform(
                    translation = (-0.4615*1.2, 0.7555*1.2),
                ),
                softening = self.softening
            ),
            "pad_3" : dl.CircularAperture(
                radius = 0.065*1.2,
                occulting = True,
                transformation=dl.CoordTransform(
                    translation = (-0.4564*1.2, -0.7606*1.2),
                ),
                softening=self.softening
            )
        }

class WFC3Optics(dl.AngularOpticalSystem):
    def __init__(self, wf_npixels, psf_npixels, psf_pixel_scale, oversample):
        super().__init__(
            wf_npixels,
            2.4,
            [
                    ("main_aperture",HSTMainAperture(transformation=dl.CoordTransform(rotation=np.pi/4), softening=2))
            ],
            psf_npixels,
            psf_pixel_scale,
            oversample
        )

from jax import lax
from dLux.psfs import PSF

class MemorySafeWFC3Optics(WFC3Optics):
    def propagate(
        self,
        wavelengths,
        offset=None,
        weights=None,
        return_wf=False,
        return_psf=False,
    ):
        if return_wf:
            raise NotImplementedError(
                "Memory-safe propagate does not return per-wavelength wavefronts, "
                "because that would require storing the full wavelength stack."
            )

        if return_wf and return_psf:
            raise ValueError("Cannot return both return_wf=True and return_psf=True.")

        wavelengths = np.atleast_1d(wavelengths)

        if weights is None:
            weights = np.ones_like(wavelengths) / len(wavelengths)
        else:
            weights = np.atleast_1d(weights)

        if offset is None:
            offset = np.zeros(2)
        else:
            offset = np.asarray(offset)

        if weights.shape != wavelengths.shape:
            raise ValueError(
                f"wavelengths and weights must have the same shape, got "
                f"{wavelengths.shape} and {weights.shape}."
            )

        # Make one monochromatic PSF to initialise the accumulator.
        # Multiply by zero so this contributes nothing.
        psf0 = np.zeros_like(
            self.propagate_mono(wavelengths[0], offset, return_wf=False)
        )

        def step(psf_acc, x):
            wavelength, weight = x

            # dLux's original code multiplies phasor by sqrt(weight),
            # then forms intensity, equivalent to weight * psf.
            psf_i = self.propagate_mono(
                wavelength,
                offset,
                return_wf=False,
            )

            return psf_acc + weight * psf_i, None

        psf, _ = lax.scan(step, psf0, (wavelengths, weights))

        if return_psf:
            # For AngularOpticalSystem this is approximately the final image pixel scale.
            # dLux normally uses wf.pixel_scale.mean(); for your purposes this is fine
            # unless you rely on the PSF object's pixel_scale very carefully.
            pixel_scale = self.psf_pixel_scale / self.oversample
            return PSF(psf, pixel_scale)

        return psf

In [ ]:
# constructing the source
N = throughput_bin.size
# throughput_bin = np.ones(N)

spectrum_model = CombinedBasisSpectrum(waves_bin, throughput_bin, np.ones(N), np.eye(N))

source_model = dl.PointSource(
    spectrum=spectrum_model
)

# for psfnpix = 512, need at least 1024 wfnpix
wf_npix = 2048
psf_npix = 512
diameter = 2.4
oversample = 4

optics = MemorySafeWFC3Optics(wf_npixels=wf_npix, psf_npixels=psf_npix, psf_pixel_scale=0.039, oversample=oversample)

detector = dl.LayeredDetector([
    (
        'downsample',
        dl.detector_layers.Downsample(kernel_size=1)
    )
])

instrument = dl.Telescope(optics, source_model, detector)

# maximum well depth: 63,000 e. 
flux = 1e6
injected_spectrum = np.ones(N)/N * flux # flat injected spectrum, same flux

img_full_wfc3 = model_psf(injected_spectrum, instrument)

source_mono = dl.PointSource(
    wavelengths=np.array([606e-9]),
    flux=flux
)

instrument_mono = dl.Telescope(optics, source_mono, detector)

img_mono_wfc3 = instrument_mono.model()

In [ ]:
aperture_wfc3 = np.load("../wfc3_aperture_2048.npy")

In [ ]:
plt.imshow(img_full_wfc3**0.25,
           cmap='cividis')

In [ ]:
def centre_crop(arr, n):
    ny, nx = arr.shape
    y0 = (ny - n) // 2
    x0 = (nx - n) // 2
    return arr[y0:y0+n, x0:x0+n], x0, y0

from matplotlib.patches import ConnectionPatch

def add_inset_connectors(ax, axins, x0, y0, n, color="white", lw=0.6):
    # edges of the source rectangle in data coords
    left   = x0 - 0.5
    right  = x0 + n - 0.5
    top    = y0 + n - 0.5
    bottom = y0 - 0.5

    # inset TOP-LEFT -> source TOP-LEFT
    con1 = ConnectionPatch(
        xyA=(0, 1),              # top-left of inset
        coordsA=axins.transAxes,
        xyB=(left, top),         # top-left of source box
        coordsB=ax.transData,
        color=color,
        linewidth=lw,
        clip_on=False,
    )

    # inset BOTTOM-RIGHT -> source BOTTOM-RIGHT
    con2 = ConnectionPatch(
        xyA=(1, 0),              # bottom-right of inset
        coordsA=axins.transAxes,
        xyB=(right, bottom),     # bottom-right of source box
        coordsB=ax.transData,
        color=color,
        linewidth=lw,
        clip_on=False,
    )

    ax.add_artist(con1)
    ax.add_artist(con2)

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

fig, axs = plt.subplots(
    2, 3,
    figsize=(fig_width, 0.66*fig_width),
)

# --------------------------------------------------
# Layout
# --------------------------------------------------

fig.subplots_adjust(
    left=0.02,
    right=0.90,     # reserve space for colourbar
    bottom=0.02,
    top=0.98,
    wspace=0.0,     # NO gap between columns
    hspace=0.00,    # small gap between rows
)

# --------------------------------------------------
# Example plotting
#
# Replace these names with your actual arrays:
#
# aperture_1, mono_psf_1, broad_psf_1
# aperture_2, mono_psf_2, broad_psf_2
# --------------------------------------------------

aperture_wfc3_padded = np.pad(aperture_wfc3, 50, mode='constant', constant_values=0)
aperture_nircam_padded = np.pad(aperture_nircam, 50, mode='constant', constant_values=0)
apertures = [aperture_wfc3_padded, aperture_nircam_padded]
mono_psfs = [img_mono_wfc3/np.max(img_mono_wfc3), img_mono/np.max(img_mono)]
broad_psfs = [img_full_wfc3/np.max(img_full_wfc3), img_full/np.max(img_full)]

# zero -> 0.5 gray
# one  -> white
aperture_cmap = ListedColormap(["0.5", "1.0"])

# for row in range(2):

#     # aperture
#     axs[row, 0].imshow(
#         apertures[row],
#         cmap=aperture_cmap,
#         vmin=0,
#         vmax=1,
#     )

#     # monochromatic PSF
#     im = axs[row, 1].imshow(
#         mono_psfs[row]**0.25,
#         cmap="cividis",
#         vmin=0,
#         vmax=1,
#     )

#     # broadband PSF
#     axs[row, 2].imshow(
#         broad_psfs[row]**0.25,
#         cmap="cividis",
#         vmin=0,
#         vmax=1,
#     )

for row in range(2):

    # aperture
    axs[row, 0].imshow(
        apertures[row],
        cmap=aperture_cmap,
        vmin=0,
        vmax=1,
    )

    # choose inset crop size for this row
    inset_n = 256 if row == 0 else 200

    # monochromatic PSF
    mono_arr = mono_psfs[row]
    im = axs[row, 1].imshow(
        mono_arr**0.125,
        cmap="magma",
        vmin=0,
        vmax=1,
    )

    mono_crop, x0, y0 = centre_crop(mono_arr, inset_n)

    axins = axs[row, 1].inset_axes([0.60, 0.60, 0.37, 0.37])  # [x0, y0, w, h] in axes coords

    axins.imshow(
        mono_crop**0.125,
        cmap="magma",
        vmin=0,
        vmax=1,
    )
    axins.set_xticks([])
    axins.set_yticks([])
    for spine in axins.spines.values():
        spine.set_color("white")
        spine.set_linewidth(0.6)

    # optional: show on main image where the inset came from
    rect = plt.Rectangle(
        (x0 - 0.5, y0 - 0.5),
        inset_n,
        inset_n,
        edgecolor="white",
        facecolor="none",
        linewidth=0.6,
    )
    axs[row, 1].add_patch(rect)
    add_inset_connectors(axs[row, 1], axins, x0, y0, inset_n)

    # broadband PSF
    broad_arr = broad_psfs[row]
    axs[row, 2].imshow(
        broad_arr**0.125,
        cmap="magma",
        vmin=0,
        vmax=1,
    )

    broad_crop, x0, y0 = centre_crop(broad_arr, inset_n)

    axins = axs[row, 2].inset_axes([0.60, 0.60, 0.37, 0.37])
    axins.imshow(
        broad_crop**0.125,
        cmap="magma",
        vmin=0,
        vmax=1,
    )
    axins.set_xticks([])
    axins.set_yticks([])
    for spine in axins.spines.values():
        spine.set_color("white")
        spine.set_linewidth(0.6)

    rect = plt.Rectangle(
        (x0 - 0.5, y0 - 0.5),
        inset_n,
        inset_n,
        edgecolor="white",
        facecolor="none",
        linewidth=0.6,
    )
    axs[row, 2].add_patch(rect)

    add_inset_connectors(axs[row, 2], axins, x0, y0, inset_n)




# --------------------------------------------------
# Axes appearance
# --------------------------------------------------

for ax in axs.flat:
    ax.set_xticks([])
    ax.set_yticks([])

    # keep all spines visible
    for spine in ax.spines.values():
        spine.set_visible(True)
        # spine.set_linewidth(0.8)

for row in range(2):
    axs[row, 0].spines["right"].set_visible(False)
    axs[row, 1].spines["right"].set_visible(False)


# --------------------------------------------------
# Colourbar spanning both rows
# --------------------------------------------------

# Original/intensity-space tick values
tick_values = np.linspace(0, 1, 7)

# Since image shown is I^(1/4), transform tick positions accordingly
tick_positions = tick_values**0.125

# Explicit colourbar axis:
# [left, bottom, width, height] in figure coordinates
top_pos = axs[0, 2].get_position()     # any top-row axis is fine
bottom_pos = axs[1, 2].get_position()  # any bottom-row axis is fine

pad = 0.015
cbar_width = 0.025

cax_left = top_pos.x1 + pad
cax_bottom = bottom_pos.y0
cax_height = top_pos.y1 - bottom_pos.y0

cax = fig.add_axes([cax_left, cax_bottom, cbar_width, cax_height])

cbar = fig.colorbar(im, cax=cax, ticks=tick_positions)

cbar.ax.set_yticklabels([
    # f"{x:g}" for x in tick_values
])
cbar.ax.tick_params(direction='in')

cbar.set_label("PSF intensity")

js.panel_label(axs[0,0], "a", x=0.04, y=0.96)

axs[0,1].text(
        0.96, 0.04,
    r'$\lambda = 606 \ \mathrm{nm}$',
    transform=axs[0,1].transAxes,
    ha='right',
    va='bottom',
    color='white'
)

axs[0,2].text(
        0.96, 0.04,
    r'F606W',
    transform=axs[0,2].transAxes,
    ha='right',
    va='bottom',
    color='white'
)

js.panel_label(axs[1,0], "b", x=0.04, y=0.96)

axs[1,1].text(
        0.96, 0.04,
    r'$\lambda = 4.44 \ \mathrm{\upmu m}$',
    transform=axs[1,1].transAxes,
    ha='right',
    va='bottom',
    color='white'
)

axs[1,2].text(
        0.96, 0.04,
    r'F444W',
    transform=axs[1,2].transAxes,
    ha='right',
    va='bottom',
    color='white'
)

plt.savefig('hubble_webb_psfs_jatis.pdf', dpi=600,
            backend='pgf')
plt.show()

In [ ]:
img_mono.shape

In [ ]:
img_mono_wfc3.shape